# Almanac1

In [7]:
import pandas as pd
import numpy as np

import time

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [9]:
df = pd.read_csv(r"C:\Users\hasin\OneDrive\Documents\Synergy-DRS\datasets\processed\almanac_final_without_last_score.xls")

print("Dataset shape:", df.shape)

Dataset shape: (29251, 334)


In [10]:
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

In [11]:
def generate_fingerprints(smiles_list):

    rows = []
    cols = []

    cache = {}

    for i, smiles in enumerate(smiles_list):

        if smiles not in cache:

            mol = Chem.MolFromSmiles(smiles)

            if mol is None:
                raise ValueError("Invalid SMILES: " + str(smiles))

            fingerprint = generator.GetFingerprint(mol)

            cache[smiles] = list(fingerprint.GetOnBits())

        for bit in cache[smiles]:

            rows.append(i)
            cols.append(bit)

    data = np.ones(len(rows), dtype=np.float32)

    return csr_matrix(
        (data, (rows, cols)),
        shape=(len(smiles_list), 2048)
    )

In [12]:
drug1_fp = generate_fingerprints(
    df["drugname1"].astype(str)
)

drug2_fp = generate_fingerprints(
    df["drugname2"].astype(str)
)

print("Drug 1 fingerprints:", drug1_fp.shape)

print("Drug 2 fingerprints:", drug2_fp.shape)

Drug 1 fingerprints: (29251, 2048)
Drug 2 fingerprints: (29251, 2048)


In [13]:
drs_columns = [
    col for col in df.columns
    if col.startswith("DRS_feature_")
]

drs_df = df[drs_columns].astype(np.float32)


In [14]:
cellline_df = pd.get_dummies(
    df["cellline"],
    prefix="Cellline",
    dtype=np.float32
)

cellline_matrix = csr_matrix(
    cellline_df.to_numpy()
)

In [15]:
X = hstack([
    csr_matrix(drs_df.to_numpy()),
    drug1_fp,
    drug2_fp,
    cellline_matrix
]).tocsr()

feature_names = (
    drs_columns
    + [f"Drug1_FP_{i}" for i in range(2048)]
    + [f"Drug2_FP_{i}" for i in range(2048)]
    + cellline_df.columns.tolist()
)

In [16]:
y = df["synergy_score"].astype(float)

print("Total input features:", X.shape[1])

print("Target shape:", y.shape)

Total input features: 4465
Target shape: (29251,)


In [20]:
print("X_train_scaled shape:", X_train_scaled.shape)

print("Target minimum:", y_train.min())
print("Target maximum:", y_train.max())
print("Target mean:", y_train.mean())
print("Target standard deviation:", y_train.std())

print("Scaler mean shape:", scaler.mean_.shape)
print("Scaler scale minimum:", scaler.scale_.min())
print("Scaler scale maximum:", scaler.scale_.max())

X_train_scaled shape: (23400, 4465)
Target minimum: -197.91693
Target maximum: 156.54813
Target mean: -20.4622570025641
Target standard deviation: 56.60824868011277
Scaler mean shape: (4465,)
Scaler scale minimum: 0.004762874868346037
Scaler scale maximum: 1.0


c:\Users\hasin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.047394e+09, tolerance: 5.011e+04
  model = cd_fast.sparse_enet_coordinate_descent(


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [18]:
scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [21]:
from sklearn.linear_model import Lasso

start_time = time.time()

lasso = Lasso(
    alpha=1.0,
    max_iter=10000,
    tol=0.001,
    selection="random"
)

lasso.fit(X_train_scaled, y_train)

print("LASSO model trained successfully!")
print("Alpha:", lasso.alpha)
print("Iterations:", lasso.n_iter_)
print("Training time:", round(time.time() - start_time, 2), "seconds")

LASSO model trained successfully!
Alpha: 1.0
Iterations: 128
Training time: 4.95 seconds


In [22]:
y_pred = lasso.predict(X_test_scaled)

In [23]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("\nALMANAC MODEL PERFORMANCE")

print("MAE:", mae)

print("MSE:", mse)

print("RMSE:", rmse)

print("R2 Score:", r2)



ALMANAC MODEL PERFORMANCE
MAE: 35.89002544011
MSE: 2396.4169104017756
RMSE: 48.953211441148326
R2 Score: 0.23171975348845375


c:\Users\hasin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.948681e+06, tolerance: 2.505e+05
  model = cd_fast.sparse_enet_coordinate_descent(
c:\Users\hasin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.363106e+07, tolerance: 2.482e+05
  model = cd_fast.sparse_enet_coordinate_descent(
c:\Users\hasin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:786: ConvergenceWarning: Objective did not converge. You might want to increase the number of itera

# DrugCombDBLincs

In [1]:
import pandas as pd
import numpy as np

import time

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(r"C:\Users\hasin\OneDrive\Documents\Synergy-DRS\datasets\processed\DrugCombDBLINCS_PRISM_final_without_last_score.csv")

print("Dataset shape:", df.shape)

Dataset shape: (17139, 334)


In [3]:
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

In [4]:
def generate_fingerprints(smiles_list):

    rows = []
    cols = []

    cache = {}

    for i, smiles in enumerate(smiles_list):

        if smiles not in cache:

            mol = Chem.MolFromSmiles(smiles)

            if mol is None:
                raise ValueError("Invalid SMILES: " + str(smiles))

            fingerprint = generator.GetFingerprint(mol)

            cache[smiles] = list(fingerprint.GetOnBits())

        for bit in cache[smiles]:

            rows.append(i)
            cols.append(bit)

    data = np.ones(len(rows), dtype=np.float32)

    return csr_matrix(
        (data, (rows, cols)),
        shape=(len(smiles_list), 2048)
    )

In [5]:
drug1_fp = generate_fingerprints(
    df["drugname1"].astype(str)
)

drug2_fp = generate_fingerprints(
    df["drugname2"].astype(str)
)

print("Drug 1 fingerprints:", drug1_fp.shape)

print("Drug 2 fingerprints:", drug2_fp.shape)

Drug 1 fingerprints: (17139, 2048)
Drug 2 fingerprints: (17139, 2048)


In [6]:
drs_columns = [
    col for col in df.columns
    if col.startswith("DRS_feature_")
]

drs_df = df[drs_columns].astype(np.float32)


In [7]:
cellline_df = pd.get_dummies(
    df["cellline"],
    prefix="Cellline",
    dtype=np.float32
)

cellline_matrix = csr_matrix(
    cellline_df.to_numpy()
)

In [8]:
X = hstack([
    csr_matrix(drs_df.to_numpy()),
    drug1_fp,
    drug2_fp,
    cellline_matrix
]).tocsr()

feature_names = (
    drs_columns
    + [f"Drug1_FP_{i}" for i in range(2048)]
    + [f"Drug2_FP_{i}" for i in range(2048)]
    + cellline_df.columns.tolist()
)

In [9]:
y = df["synergy_score"].astype(float)

print("Total input features:", X.shape[1])

print("Target shape:", y.shape)

Total input features: 4491
Target shape: (17139,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [12]:
from sklearn.linear_model import Lasso
import time

start_time = time.time()

lasso = Lasso(
    alpha=0.05,
    max_iter=5000,
    tol=0.01,
    selection="random",
    random_state=42
)

lasso.fit(X_train_scaled, y_train)

print("LASSO model trained successfully!")
print("Alpha:", lasso.alpha)
print("Iterations:", lasso.n_iter_)
print("Training time:", round(time.time() - start_time, 2), "seconds")

LASSO model trained successfully!
Alpha: 0.05
Iterations: 46
Training time: 0.46 seconds


In [13]:
y_pred = lasso.predict(X_test_scaled)

In [14]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("\nALMANAC MODEL PERFORMANCE")

print("MAE:", mae)

print("MSE:", mse)

print("RMSE:", rmse)

print("R2 Score:", r2)



ALMANAC MODEL PERFORMANCE
MAE: 3.3858891638758024
MSE: 22.527308127092933
RMSE: 4.746294146709929
R2 Score: 0.2191891177565325


# DrugCombLincs1

In [2]:

# STEP 1: Import libraries

import pandas as pd
import numpy as np

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# STEP 2: Define all five datasets

datasets = {
    "OncologyScreenLINCS_PRISM": "OncologyScreenLINCS_PRISM_final_without_last_score.xls",

    "OneilLINCS_PRISM": "OneilLINCS_PRISM_final_without_last_score.csv",

    "DrugComb_LINCS_PRISM": "DrugComb_LINCS_PRISM_final_without_last_score.csv",

    "DrugCombDBLINCS_PRISM": "DrugCombDBLINCS_PRISM_final_without_last_score.csv",

    "AlmanacLINCS_PRISM": "almanac_final_without_last_score.xls"
}


# STEP 3: Create Morgan Fingerprint Generator

generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)


# STEP 4: Function to generate Morgan fingerprints

def generate_fingerprints(smiles_list):

    rows = []
    cols = []

    cache = {}

    for i, smiles in enumerate(smiles_list):

        if smiles not in cache:

            mol = Chem.MolFromSmiles(smiles)

            if mol is None:
                raise ValueError("Invalid SMILES: " + str(smiles))

            fp = generator.GetFingerprint(mol)

            cache[smiles] = list(fp.GetOnBits())

        for bit in cache[smiles]:

            rows.append(i)
            cols.append(bit)

    data = np.ones(len(rows), dtype=np.float32)

    return csr_matrix(
        (data, (rows, cols)),
        shape=(len(smiles_list), 2048)
    )


# STEP 5: Train each dataset one after another

for dataset_name, file_path in datasets.items():

    print("\n" + "=" * 60)
    print("PROCESSING:", dataset_name)
    print("=" * 60)

    # Load dataset (all five files are CSV-formatted)

    df = pd.read_csv(file_path)

    print("Dataset shape:", df.shape)


    # Separate input features and target

    y = df["synergy_score"].astype(float)

    drs_columns = [
        col for col in df.columns
        if col.startswith("DRS_feature_")
    ]

    if len(drs_columns) != 330:
        raise ValueError("Expected 330 DRS features.")

    drs_df = df[drs_columns].astype(np.float32)


    # Generate Morgan fingerprints for both drugs

    drug1_fp = generate_fingerprints(
        df["drugname1"].astype(str)
    )

    drug2_fp = generate_fingerprints(
        df["drugname2"].astype(str)
    )


    # Encode cell-line information

    cellline_df = pd.get_dummies(
        df["cellline"],
        prefix="Cellline",
        dtype=np.float32
    )

    cellline_matrix = csr_matrix(
        cellline_df.to_numpy()
    )


    # Combine all input features for THIS dataset only

    X = hstack([
        csr_matrix(drs_df.to_numpy()),
        drug1_fp,
        drug2_fp,
        cellline_matrix
    ]).tocsr()

    feature_names = (
        drs_columns
        + [f"Drug1_FP_{i}" for i in range(2048)]
        + [f"Drug2_FP_{i}" for i in range(2048)]
        + cellline_df.columns.tolist()
    )

    print("Total input features:", X.shape[1])


    # Split training and testing data

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )


    # Standardize features

    scaler = StandardScaler(with_mean=False)

    X_train_scaled = scaler.fit_transform(X_train)

    X_test_scaled = scaler.transform(X_test)


    # Train LASSO Regression

    lasso = LassoCV(
        cv=5,
        max_iter=100000,
        n_jobs=-1
    )

    lasso.fit(X_train_scaled, y_train)

    print("\nLASSO model trained successfully!")

    print("Best Alpha:", lasso.alpha_)


    # Make predictions

    y_pred = lasso.predict(X_test_scaled)


    # Calculate performance metrics

    mae = mean_absolute_error(y_test, y_pred)

    mse = mean_squared_error(y_test, y_pred)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, y_pred)


    # Display results after completing this dataset

    print("\nRESULTS FOR:", dataset_name)

    print("--------------------------------")

    print("MAE:", mae)

    print("MSE:", mse)

    print("RMSE:", rmse)

    print("R2 Score:", r2)


    # Display selected features

    coefficients = pd.Series(
        lasso.coef_,
        index=feature_names
    )

    selected_features = coefficients[
        coefficients != 0
    ]

    print("\nNumber of selected features:", len(selected_features))

    print("\nTop 20 selected features:")

    print(
        selected_features.sort_values(
            key=abs,
            ascending=False
        ).head(20)
    )

    print("\nCompleted:", dataset_name)

    print("=" * 60)


print("\nALL FIVE DATASETS PROCESSED!")



PROCESSING: OncologyScreenLINCS_PRISM


FileNotFoundError: [Errno 2] No such file or directory: 'OncologyScreenLINCS_PRISM_final_without_last_score.xls'

# OncologyScreenLINCS

In [1]:
import pandas as pd
import numpy as np

import time

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(r"C:\Users\hasin\OneDrive\Documents\Synergy-DRS\datasets\processed\DrugCombDBLINCS_PRISM_final_without_last_score.csv")

print("Dataset shape:", df.shape)

Dataset shape: (17139, 334)


In [3]:
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

In [4]:
def generate_fingerprints(smiles_list):

    rows = []
    cols = []

    cache = {}

    for i, smiles in enumerate(smiles_list):

        if smiles not in cache:

            mol = Chem.MolFromSmiles(smiles)

            if mol is None:
                raise ValueError("Invalid SMILES: " + str(smiles))

            fingerprint = generator.GetFingerprint(mol)

            cache[smiles] = list(fingerprint.GetOnBits())

        for bit in cache[smiles]:

            rows.append(i)
            cols.append(bit)

    data = np.ones(len(rows), dtype=np.float32)

    return csr_matrix(
        (data, (rows, cols)),
        shape=(len(smiles_list), 2048)
    )

In [5]:
drug1_fp = generate_fingerprints(
    df["drugname1"].astype(str)
)

drug2_fp = generate_fingerprints(
    df["drugname2"].astype(str)
)

print("Drug 1 fingerprints:", drug1_fp.shape)

print("Drug 2 fingerprints:", drug2_fp.shape)

Drug 1 fingerprints: (17139, 2048)
Drug 2 fingerprints: (17139, 2048)


In [6]:
drs_columns = [
    col for col in df.columns
    if col.startswith("DRS_feature_")
]

drs_df = df[drs_columns].astype(np.float32)


In [7]:
cellline_df = pd.get_dummies(
    df["cellline"],
    prefix="Cellline",
    dtype=np.float32
)

cellline_matrix = csr_matrix(
    cellline_df.to_numpy()
)

In [8]:
X = hstack([
    csr_matrix(drs_df.to_numpy()),
    drug1_fp,
    drug2_fp,
    cellline_matrix
]).tocsr()

feature_names = (
    drs_columns
    + [f"Drug1_FP_{i}" for i in range(2048)]
    + [f"Drug2_FP_{i}" for i in range(2048)]
    + cellline_df.columns.tolist()
)

In [9]:
y = df["synergy_score"].astype(float)

print("Total input features:", X.shape[1])

print("Target shape:", y.shape)

Total input features: 4491
Target shape: (17139,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [12]:
from sklearn.linear_model import Lasso
import time

start_time = time.time()

lasso = Lasso(
    alpha=0.05,
    max_iter=5000,
    tol=0.01,
    selection="random",
    random_state=42
)

lasso.fit(X_train_scaled, y_train)

print("LASSO model trained successfully!")
print("Alpha:", lasso.alpha)
print("Iterations:", lasso.n_iter_)
print("Training time:", round(time.time() - start_time, 2), "seconds")

LASSO model trained successfully!
Alpha: 0.05
Iterations: 46
Training time: 0.55 seconds


In [13]:
y_pred = lasso.predict(X_test_scaled)

In [14]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("\nALMANAC MODEL PERFORMANCE")

print("MAE:", mae)

print("MSE:", mse)

print("RMSE:", rmse)

print("R2 Score:", r2)



ALMANAC MODEL PERFORMANCE
MAE: 3.3858891638758024
MSE: 22.527308127092933
RMSE: 4.746294146709929
R2 Score: 0.2191891177565325


# OneilLINCS

In [1]:
import pandas as pd
import numpy as np

import time

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pd.read_csv(r"C:\Users\hasin\OneDrive\Documents\Synergy-DRS\datasets\processed\OneilLINCS_PRISM_final_without_last_score.csv")

print("Dataset shape:", df.shape)

Dataset shape: (2714, 334)


In [3]:
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048
)

In [4]:
def generate_fingerprints(smiles_list):

    rows = []
    cols = []

    cache = {}

    for i, smiles in enumerate(smiles_list):

        if smiles not in cache:

            mol = Chem.MolFromSmiles(smiles)

            if mol is None:
                raise ValueError("Invalid SMILES: " + str(smiles))

            fingerprint = generator.GetFingerprint(mol)

            cache[smiles] = list(fingerprint.GetOnBits())

        for bit in cache[smiles]:

            rows.append(i)
            cols.append(bit)

    data = np.ones(len(rows), dtype=np.float32)

    return csr_matrix(
        (data, (rows, cols)),
        shape=(len(smiles_list), 2048)
    )

In [5]:
drug1_fp = generate_fingerprints(
    df["drugname1"].astype(str)
)

drug2_fp = generate_fingerprints(
    df["drugname2"].astype(str)
)

print("Drug 1 fingerprints:", drug1_fp.shape)

print("Drug 2 fingerprints:", drug2_fp.shape)

Drug 1 fingerprints: (2714, 2048)
Drug 2 fingerprints: (2714, 2048)


In [6]:
drs_columns = [
    col for col in df.columns
    if col.startswith("DRS_feature_")
]

drs_df = df[drs_columns].astype(np.float32)


In [7]:
cellline_df = pd.get_dummies(
    df["cellline"],
    prefix="Cellline",
    dtype=np.float32
)

cellline_matrix = csr_matrix(
    cellline_df.to_numpy()
)

In [8]:
X = hstack([
    csr_matrix(drs_df.to_numpy()),
    drug1_fp,
    drug2_fp,
    cellline_matrix
]).tocsr()

feature_names = (
    drs_columns
    + [f"Drug1_FP_{i}" for i in range(2048)]
    + [f"Drug2_FP_{i}" for i in range(2048)]
    + cellline_df.columns.tolist()
)

In [9]:
y = df["synergy_score"].astype(float)

print("Total input features:", X.shape[1])

print("Target shape:", y.shape)

Total input features: 4458
Target shape: (2714,)


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [11]:
scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [12]:
from sklearn.linear_model import Lasso
import time

start_time = time.time()

lasso = Lasso(
    alpha=0.05,
    max_iter=5000,
    tol=0.01,
    selection="random",
    random_state=42
)

lasso.fit(X_train_scaled, y_train)

print("LASSO model trained successfully!")
print("Alpha:", lasso.alpha)
print("Iterations:", lasso.n_iter_)
print("Training time:", round(time.time() - start_time, 2), "seconds")

LASSO model trained successfully!
Alpha: 0.05
Iterations: 61
Training time: 0.15 seconds


In [13]:
y_pred = lasso.predict(X_test_scaled)

In [14]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("\nALMANAC MODEL PERFORMANCE")

print("MAE:", mae)

print("MSE:", mse)

print("RMSE:", rmse)

print("R2 Score:", r2)



ALMANAC MODEL PERFORMANCE
MAE: 14.234726930005333
MSE: 356.30449094509913
RMSE: 18.876029533381725
R2 Score: 0.28856536693703794
